# Example - Engineering Portal 
Demonstrates self-describing portals between engineering tools.

**Scenario**: 
> A SysML v2 model tool needs to send design parameters to a
simulation tool.  The simulation tool's SHACL shapes declare exactly
what it accepts.  The SysML tool queries those shapes, maps its own
properties, and generates the portal CONSTRUCT automatically.

This is the "*self-describing portal surface*" pattern; the target's
membrane IS the API documentation, and the portal query IS generated
from that documentation.

In [ ]:
from holonic import (
    Holon, TransformPortal, Holarchy,
    validate_membrane, MembraneHealth,
    ProvenanceTracker,
    discover_target_shape, generate_construct_query, describe_surface,
)

## Build the SysML modeler holon

In [ ]:
sysml = Holon(
    iri="urn:holon:tool:sysml",
    label="SysML v2 Modeler",
    depth=1,
    interior_ttl="""
        @prefix sysml: <urn:sysml:> .

        <urn:sysml:block:thermal-mgmt> a sysml:Block ;
            sysml:name "ThermalMgmtSubsystem" ;
            sysml:owner <urn:sysml:pkg:vehicle-systems> .

        <urn:sysml:param:mass> a sysml:ValueProperty ;
            sysml:name       "mass" ;
            sysml:value      142.3 ;
            sysml:unit       "kg" ;
            sysml:memberOf   <urn:sysml:block:thermal-mgmt> .

        <urn:sysml:param:power> a sysml:ValueProperty ;
            sysml:name       "maxPowerDraw" ;
            sysml:value      2.8 ;
            sysml:unit       "kW" ;
            sysml:memberOf   <urn:sysml:block:thermal-mgmt> .

        <urn:sysml:param:flow> a sysml:ValueProperty ;
            sysml:name       "coolantFlowRate" ;
            sysml:value      15.0 ;
            sysml:unit       "L/min" ;
            sysml:memberOf   <urn:sysml:block:thermal-mgmt> .

        <urn:sysml:port:inlet> a sysml:FlowPort ;
            sysml:name       "coolantInlet" ;
            sysml:direction  "in" ;
            sysml:memberOf   <urn:sysml:block:thermal-mgmt> .
    """,
    boundary_ttl="""
        @prefix sysml: <urn:sysml:> .

        <urn:shapes:sysml:BlockShape> a sh:NodeShape ;
            sh:targetClass sysml:Block ;
            sh:property [
                sh:path sysml:name ;
                sh:minCount 1 ; sh:maxCount 1 ;
                sh:datatype xsd:string ;
                sh:severity sh:Violation ;
                sh:message "Block must have a name."
            ] .

        <urn:shapes:sysml:ValuePropertyShape> a sh:NodeShape ;
            sh:targetClass sysml:ValueProperty ;
            sh:property [
                sh:path sysml:name ;
                sh:minCount 1 ;
                sh:datatype xsd:string ;
                sh:severity sh:Violation
            ] ;
            sh:property [
                sh:path sysml:value ;
                sh:minCount 1 ;
                sh:severity sh:Violation ;
                sh:message "Value property must have a value."
            ] ;
            sh:property [
                sh:path sysml:unit ;
                sh:minCount 1 ;
                sh:datatype xsd:string ;
                sh:severity sh:Warning ;
                sh:message "Value property should declare units."
            ] .
    """,
)

print(sysml)

## Build the simulation tool holon with SHACL-defined input surface

In [ ]:
sim = Holon(
    iri="urn:holon:tool:sim",
    label="Simulation Tool",
    depth=1,

    # Boundary: the self-describing surface.
    # THESE SHAPES ARE THE API CONTRACT.
    # Anyone can query them to discover what the sim accepts.
    boundary_ttl="""
        @prefix sim: <urn:sim:> .

        <urn:shapes:sim:SimInputShape> a sh:NodeShape ;
            sh:targetClass sim:InputBlock ;
            sh:property [
                sh:path rdfs:label ;
                sh:minCount 1 ;
                sh:maxCount 1 ;
                sh:datatype xsd:string ;
                sh:severity sh:Violation ;
                sh:message "Input block must have a label."
            ] ;
            sh:property [
                sh:path sim:inputParam ;
                sh:minCount 1 ;
                sh:severity sh:Violation ;
                sh:message "Input block must have at least one parameter."
            ] .

        <urn:shapes:sim:SimParamShape> a sh:NodeShape ;
            sh:targetClass sim:InputParam ;
            sh:property [
                sh:path rdfs:label ;
                sh:minCount 1 ;
                sh:datatype xsd:string ;
                sh:severity sh:Violation ;
                sh:message "Parameter must have a label."
            ] ;
            sh:property [
                sh:path sim:paramValue ;
                sh:minCount 1 ;
                sh:severity sh:Violation ;
                sh:message "Parameter must have a numeric value."
            ] ;
            sh:property [
                sh:path sim:paramUnit ;
                sh:minCount 1 ;
                sh:datatype xsd:string ;
                sh:severity sh:Warning ;
                sh:message "Parameter should specify units."
            ] .
    """,
)

print(sim)

## Discover the simulation tool's surface

In [ ]:
print(describe_surface(sim))

shapes = discover_target_shape(sim)
print("\n  Structured discovery results:")
for cls, props in shapes.items():
    cls_short = cls.split(":")[-1] if ":" in cls else cls.split("/")[-1]
    print(f"\n  Class: {cls_short}")
    for p in props:
        req = "REQUIRED" if p.is_required else "optional"
        dt = p.datatype.split("#")[-1] if p.datatype else "any"
        print(f"    {p.path_local:<20s} {req:<10s} [{dt}]  {p.severity}")

## Generate the portal CONSTRUCT from the discovered surface

Two CONSTRUCT queries needed:
 1. Block → InputBlock (with inputParam links)
 2. ValueProperty → InputParam

In [ ]:
# The block-level query
block_query = """
PREFIX sysml: <urn:sysml:>
PREFIX sim:   <urn:sim:>
PREFIX rdfs:  <http://www.w3.org/2000/01/rdf-schema#>

CONSTRUCT {
    ?block a sim:InputBlock ;
        rdfs:label ?name ;
        sim:inputParam ?param .
    
    ?param a sim:InputParam ;
        rdfs:label ?pname ;
        sim:paramValue ?val ;
        sim:paramUnit ?unit .
}
WHERE {
    ?block a sysml:Block ;
        sysml:name ?name .
        
    ?param sysml:memberOf ?block ;
        a sysml:ValueProperty ;
        sysml:name ?pname ;
        sysml:value ?val .
    
    OPTIONAL { ?param sysml:unit ?unit }
}
"""

## Create and traverse the portal

In [ ]:
portal = TransformPortal(
    iri="urn:portal:sysml-to-sim",
    source=sysml,
    target=sim,
    label="SysML → Simulation Input",
    construct_query=block_query,
    validate_output=False,
)

projected = portal.traverse(sysml.interior)

print(f"  Source interior: {len(sysml.interior)} triples")
print(f"  Projected output: {len(projected)} triples\n")

print("  Projected triples (simulation-ready):\n")
for s, p, o in sorted(projected):
    s_short = str(s).split(":")[-1]
    p_short = (str(p)
               .replace("urn:sim:", "sim:")
               .replace("http://www.w3.org/2000/01/rdf-schema#", "rdfs:")
               .replace("http://www.w3.org/1999/02/22-rdf-syntax-ns#", "rdf:"))
    print(f"    {s_short:35s} {p_short:20s} {o.n3()}")

## Validate projected output against target membrane

In [ ]:
# Load projected data into sim's interior
for t in projected:
    sim.interior.add(t)

result = validate_membrane(sim)
print(result.summary())

## Provenance

In [ ]:
prov = ProvenanceTracker(
    agent_iri="urn:agent:digital-thread-mgr",
    agent_label="Digital Thread Manager",
)

activity = prov.record_traversal(
    portal_iri=portal.iri,
    source=sysml,
    target=sim,
    notes="Translated SysML v2 design model to simulation input format.",
)

prov.record_validation(
    holon=sim,
    conforms=result.conforms,
    health=result.health.value,
)

print(f"  Traversal activity: {activity}")
print(f"  Target context graph: {len(sim.context)} triples\n")

# Show PROV-O triples
print("  Provenance trail:\n")
prov_q = """
PREFIX prov: <http://www.w3.org/ns/prov#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?activity ?label ?time
WHERE {
    ?activity a prov:Activity .
    ?activity rdfs:label ?label .
    ?activity prov:startedAtTime ?time .
}
ORDER BY ?time
"""
for row in sim.context.query(prov_q):
    print(f"    {row[1]}")
    print(f"      at: {row[2]}")

## Summary

The pattern demonstrated here:

  1. TARGET declares its boundary shapes (SHACL NodeShapes)</br>
     → "I accept sim:InputBlock with these properties"
  2. SOURCE queries the target's shapes (discover_target_shape)</br>
     → "The target needs: label (required), inputParam (required), ..."
  3. ENGINEER writes the property map (domain knowledge)</br>
     → sysml:name → rdfs:label, sysml:value → sim:paramValue, ...
  4. LIBRARY generates the SPARQL CONSTRUCT (generate_construct_query)</br>
     → The translation query is derived from the shape + the map
  5. PORTAL carries the CONSTRUCT and executes it on traversal</br>
     → Source interior triples → projected target-shaped triples
  6. TARGET validates the projected output against its own shapes</br>
     → The membrane accepts or rejects the incoming data
  7. PROVENANCE records the entire operation in the context graph</br>
     → Who, when, what was used, what was generated

The target's SHACL shapes are SIMULTANEOUSLY:
    - A validation rulebook (what data is valid inside)
    - An API contract (what projections must look like)
    - A discovery surface (what the source can query to learn the shape)
    - A documentation spec (human-readable via describe_surface())

This is what Cagel means by "the membrane is not just a container;
 it is an active participant in the cell's identity.